In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [3]:
# data
iris = load_iris()
X, y = pd.DataFrame(iris.data, columns=iris.feature_names), pd.Series(iris.target, name='species')

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 

# pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
])

model = pipeline.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
y_pred_proba = model.predict_proba(X_test)

In [8]:
svm_model = pipeline.named_steps['classifier']

print("\n" + "="*60)
print("SUPPORT VECTORS (The most important data points)")
print("="*60)
print(f"Number of Support Vectors: {len(svm_model.support_vectors_)}")
print(f"Support Vector Indices: {svm_model.support_}")
print(f"Support Vector Count per Class: {svm_model.n_support_}")
print("\nFirst 5 Support Vectors (scaled):")
print(svm_model.support_vectors_[:5].round(3))


SUPPORT VECTORS (The most important data points)
Number of Support Vectors: 47
Support Vector Indices: [  0  19  23  33  40  48  60  71  83  93   2  11  12  20  37  39  41  51
  61  64  67  68  79  82  87  88  97 118   1   5   7  17  18  22  25  28
  35  46  55  59  73  75  78  91  95  98 102]
Support Vector Count per Class: [10 18 19]

First 5 Support Vectors (scaled):
[[-1.722 -0.332 -1.346 -1.323]
 [-0.527  0.787 -1.289 -1.06 ]
 [-1.005 -0.108 -1.232 -1.323]
 [-0.169  3.026 -1.289 -1.06 ]
 [-1.483  0.787 -1.346 -1.192]]


In [4]:
target_names = iris.target_names

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")
print()

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("\n" + "="*60)
print("CROSS-VALIDATION")
print("="*60)
print(f"CV Scores: {cv_scores}")
print(f"CV Mean: {cv_scores.mean():.4f}")
print(f"CV Std: {cv_scores.std():.4f}")


MODEL EVALUATION
Training Accuracy: 97.50%
Test Accuracy: 96.67%
Overfitting Gap: 0.0083
✅ NO OVERFITTING (gap < 1%)
✅ No underfitting detected

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
Correct predictions: 29
Total predictions: 30
Accuracy: 0.9667
Accuracy: 96.67%

setosa Accuracy: 1.0000 (100.00%)
versicolor Accuracy: 0.9000 (90.00%)
virginica Accuracy: 1.0000 (100.00%)

CROSS-VALIDATION
CV Scores: [0.91666667 1.         0.95833333 0.95833333 1.        ]
CV Mean: 0.9667
CV Std: 0.0312


In [6]:
print("\n" + "="*60)
print("HYPERPARAMETER TUNING (Grid Search)")
print("="*60)

param_grid = {
    'classifier__C': [0.1, 1, 10, 100],    
    'classifier__gamma': ['scale', 'auto', 0.1, 1], 
    'classifier__kernel': ['rbf', 'poly', 'sigmoid'] 
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Evaluate best model
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_best)

print(f"Test Accuracy with best params: {test_accuracy:.4f}")


HYPERPARAMETER TUNING (Grid Search)
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters: {'classifier__C': 1, 'classifier__gamma': 0.1, 'classifier__kernel': 'rbf'}
Best CV Score: 0.9750
Test Accuracy with best params: 0.9667
